In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.models import DenseNet121_Weights, densenet121, EfficientNet_B2_Weights, efficientnet_b2, resnet34, ResNet34_Weights
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import pydicom
import torch.nn as nn
import torch.optim as optim

In [2]:
image = pydicom.dcmread('Dataset/rsna-pneumonia-detection-challenge/stage_2_train_images/0a0f91dc-6015-4342-b809-d19610854a21.dcm')
image.pixel_array

array([[246, 245, 241, ..., 231, 252, 153],
       [242, 242, 239, ..., 231, 251, 151],
       [239, 239, 237, ..., 230, 250, 150],
       ...,
       [151, 149, 149, ..., 215, 243, 151],
       [148, 146, 145, ..., 215, 243, 150],
       [146, 145, 146, ..., 217, 244, 151]],
      shape=(1024, 1024), dtype=uint8)

In [4]:
class RSNA_Dataset(Dataset):
    def __init__(self, data, transform):
        self.data = data
        self.transform = transform


    def __len__(self):
        return len(self.data)

    def __getitem__(self, item):
        current = self.data.iloc[item]
        patientId = current['patientId']
        path = f'Dataset/rsna-pneumonia-detection-challenge/stage_2_train_images/{patientId}.dcm'
        data = pydicom.dcmread(path)
        array = data.pixel_array
        label = current['Target']
        if self.transform:
            array = self.transform(array)
        return (array, label)


def train_transform(resize, mean, std, crop_size):
    transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(size=crop_size),
    transforms.RandomHorizontalFlip(0.1),
    transforms.RandomRotation(degrees=(-10, +10)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize(size=resize),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)])
    return transform

def val_transform(resize, mean, std):
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize(size=resize),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std= std)
    ])
    return transform

In [5]:
data = pd.read_csv('Dataset/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv')
train, val = train_test_split(data, test_size=0.3, shuffle=True)
dense_train_transform = train_transform(resize=[256], crop_size=[224], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
dense_val_transform = val_transform(resize=[256], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_dataset = RSNA_Dataset(data=train, transform=dense_train_transform)
val_dataset = RSNA_Dataset(data=val, transform=dense_val_transform)

In [6]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)
eval_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=64)


In [7]:
dense_model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
dense_model.classifier = nn.Linear(in_features=1024, out_features=2)
for param in dense_model.parameters():
    param.requires_grad = False

for param in dense_model.classifier.parameters():
    param.requires_grad=True


optimization = optim.Adam(dense_model.classifier.parameters(), lr=0.0001)
lr_sh = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimization, patience=2, factor=0.1, mode='max')
loss_function = nn.CrossEntropyLoss()

In [11]:
def evaluate(model, loss_function, val_dataloader, train:bool):
    model.eval()
    val_loss = 0.0
    real_values = []
    predicted = []
    with torch.no_grad():
        for image, label in val_dataloader:
            image, label = image.to(device), label.to(device)
            predictions =model(image)
            loss = loss_function(predictions, label)
            val_loss += loss
            classification = torch.argmax(predictions, dim=1)
            real_values.extend(label.tolist())
            predicted.extend(classification.tolist())

    report = classification_report(real_values, predicted, output_dict=True)
    f1  =  report['1']['f1-score']
    if train:
        return (val_loss.item())/len(val_dataloader), f1
    return (val_loss.item())/len(val_dataloader), f1, real_values, predicted


In [12]:
#training loop
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
dense_model = dense_model.to(device)
def training_loop(model, loss_function , optimization, lr_sch, model_name:str, train_dataloader):
    device = torch.device('mps' if torch.mps.is_available() else 'cpu')
    model = model.to(device)
    model.train()
    patience = 4
    counter = 0
    best_f1 = 0.0

    for i in range(0, 30):
        running_loss  = 0.0
        training_loss = 0.0
        for image, label in train_dataloader:
            image, label = image.to(device), label.to(device)
            preditions = model(image)
            loss = loss_function(preditions, label)
            running_loss += loss.item()
            optimization.zero_grad()
            loss.backward()
            optimization.step()
        training_loss = running_loss/len(train_dataloader)
        val_loss , f1 = evaluate(dense_model, loss_function, eval_dataloader, train=True)
        if best_f1< f1 :
            best_f1 = f1
            counter = 0
            print("New model is saved")
            torch.save(dense_model.state_dict(), f'{model_name}_bestweights.pth')
        else:
            counter +=1
        if counter >=patience:
            print("Early Stopping")
            break
        lr_sh.step(f1)
        print(f'For the Epoch {i+1}\n'
          f'Training loss : {training_loss:.4f}\n'
          f'Validation loss : {val_loss:.4f}\n'
          f'F1 score is {f1}')
    return model




In [14]:
dense_model= training_loop(model=dense_model, loss_function=loss_function, optimization=optimization, lr_sch=lr_sh, model_name='dense',train_dataloader=train_dataloader)
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
dense_model = dense_model.to(device)

In [17]:
dense_model.load_state_dict(torch.load('models/dense_model.pth', weights_only=True, map_location='mps'))

<All keys matched successfully>

In [90]:
val , f1 = evaluate(dense_model, loss_function, eval_dataloader)

In [8]:

model = 'dense'

In [19]:
torch.save(dense_model.state_dict(), '_bestweights.pth')

In [ ]:
DataLoader(batch_size=64, )